# Qwen3-14B SFT evaluation


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/devaanshpa/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path("/kaggle/working/CrashDiag")
if WORKDIR.exists():
    subprocess.run(["rm", "-rf", str(WORKDIR)], check=True)
subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BASE_MODEL = "Qwen/Qwen3-14B"
MODEL_SLUG = "qwen3_14b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{MODEL_SLUG}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"base_model={BASE_MODEL}")
print(f"dataset_run_id={DATASET_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

SFT_RUN_ID = os.environ.get("CRASHDIAG_SFT_RUN_ID", "").strip()
SFT_EVAL_RUN_ID = os.environ.get("CRASHDIAG_SFT_EVAL_RUN_ID") or ist_run_id("sft-eval")
if not SFT_RUN_ID: raise RuntimeError("Set CRASHDIAG_SFT_RUN_ID to the completed SFT run ID.")
DATASET_DIR, SFT_DIR = Path("artifacts/datasets"), Path("artifacts/sft")
ArtifactUploader(ArtifactConfig.from_env(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID)).download_stage("datasets", DATASET_DIR)
ArtifactUploader(ArtifactConfig.from_env(bucket_id=BUCKET_ID, run_id=SFT_RUN_ID)).download_stage("sft", SFT_DIR)


In [ ]:
from training.evaluate_jsonl import main as evaluate_main

exit_code = evaluate_main([
    "--model", str(SFT_DIR), "--dataset", str(DATASET_DIR / "grpo_eval.jsonl"),
    "--output-dir", "outputs/sft-eval", "--load-in-4bit", "--precision", "fp16",
    "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
    "--artifact-bucket", BUCKET_ID, "--artifact-run-id", SFT_EVAL_RUN_ID, "--artifact-stage", "sft-eval",
])
if exit_code: raise RuntimeError(f"SFT evaluation failed: {exit_code}")
